### Transform Customer Data
- Remove records with NULL Customer_id
- Remove exact duplicate reords
- Remove duplicates based on the created_timesatmp
- CAST the columns to correct datatypes
- Write transformed data to the silver schema

#### 1. Remove records with NULL Customer_id

In [0]:
%sql
SELECT
    *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL;

#### 2. Remove exact duplicate reords

In [0]:
%sql
SELECT
    DISTINCT *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

#### 3. Remove duplicates based on the created_timesatmp

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_customers_distinct AS
SELECT
    DISTINCT *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
WITH cte_max_cust_time AS (
    SELECT
        customer_id,
        MAX(created_timestamp) AS max_created_timestamp
    FROM gizmobox.bronze.v_customers
    GROUP BY customer_id
)
SELECT
    *
FROM v_customers_distinct AS t
INNER JOIN cte_max_cust_time AS cte 
ON t.customer_id = cte.customer_id AND t.created_timestamp = cte.max_created_timestamp
ORDER BY t.customer_id;
  

####4. CAST the columns to correct datatypes

In [0]:
%sql
WITH cte_max_cust_time AS (
    SELECT
        customer_id,
        MAX(created_timestamp) AS max_created_timestamp
    FROM gizmobox.bronze.v_customers
    GROUP BY customer_id
)
SELECT
    CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
    t.customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) AS date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) AS member_since,
    t.telephone,
    t.file_path
FROM v_customers_distinct AS t
INNER JOIN cte_max_cust_time AS cte 
ON t.customer_id = cte.customer_id AND t.created_timestamp = cte.max_created_timestamp
ORDER BY t.customer_id;

#### 5. Write transformed data to the silver schema

In [0]:
%sql
CREATE TABLE gizmobox.silver.customers
AS
WITH cte_max_cust_time AS (
    SELECT
        customer_id,
        MAX(created_timestamp) AS max_created_timestamp
    FROM gizmobox.bronze.v_customers
    GROUP BY customer_id
)
SELECT
    CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
    t.customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) AS date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) AS member_since,
    t.telephone,
    t.file_path
FROM v_customers_distinct AS t
INNER JOIN cte_max_cust_time AS cte 
ON t.customer_id = cte.customer_id AND t.created_timestamp = cte.max_created_timestamp;

In [0]:
%sql
SELECT * FROM gizmobox.silver.customers;

In [0]:
%sql
DESCRIBE EXTENDED gizmobox.silver.customers;
    
